# Declare a new run

Bare-minimum template for starting a new run: define its sources, its
tokenizer (reused if one already matches), a dataset and a pretraining config
under a fresh `run_id`, then `check()` and `write()` against a root.

Copy this notebook per run and change `RUN_ID` and the knobs below.
Everything past declaring -- running jobs, loading artifacts back,
visualizing a plan, conflicts -- is `demo.ipynb`, not repeated here.

In [15]:
from pathlib import Path

from artifact import (
    DataSet,
    ModelParameters,
    Pretraining,
    PretrainingConfig,
    Source,
    Tokenizer,
)
from resolve import Declaration

root = Path("../storage")  # kept separate from the real /data tree

## Sources

In [2]:
odyssey = Source(
    name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
)
mobydick = Source(
    name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
)
romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)
montecristo = Source(
    name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt"
)

## The run

In [22]:
RUN_ID = "my-run2"  # <- change this per run

# same tokenizer any other run trained on these sources would build -- if one's
# already declared, it's reused rather than rebuilt from scratch
tokenizer = Tokenizer(
    vocab_size=300,
    kind="bpe",
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick),
)

dataset = DataSet.from_sources(
    run_id=RUN_ID,
    tokenizer=tokenizer,
    train_sources=[odyssey, mobydick],
    valid_sources=[romeojuliet, montecristo],
)

model_parameters = ModelParameters(hidden_size=64, num_layers=2)
config = PretrainingConfig(
    total_steps=2000, batch_size=32, lr=1e-3, seed=1, checkpoint_every=500
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=dataset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    config=config,
)

pretraining  # parameters all the way down; `commit` is hidden from the repr

Pretraining(run_id='my-run2', dataset=DataSet(run_id='my-run2', train_set=(TokenizedSource(tokenizer=Tokenizer(vocab_size=300, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt')), kind='bpe'), source=Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt')), TokenizedSource(tokenizer=Tokenizer(vocab_size=300, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt')), kind='bpe'), source=Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt'))), valid_set=(TokenizedSource(tokenizer=Tokenizer(vocab_size=300, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg17

## Declare

`check()` resolves the request and reconciles it against `root` without
writing anything -- everything shared (sources, and the tokenizer if it
matches one already declared) should read `done`; everything new to this run
should read `new`. `write()` declares whatever's `new`, refusing outright if
anything's inconsistent.

In [23]:
declaration = Declaration(pretraining, root)
print(declaration.check())

run my-run2 under ../storage
  declared   sources/odyssey
  declared   sources/mobydick
  new        tokenizers/bpe-300-e4649eb4ff
  new        tokenizers/bpe-300-e4649eb4ff/bin/odyssey
  new        tokenizers/bpe-300-e4649eb4ff/bin/mobydick
  declared   sources/romeojuliet
  new        tokenizers/bpe-300-e4649eb4ff/bin/romeojuliet
  declared   sources/montecristo
  new        tokenizers/bpe-300-e4649eb4ff/bin/montecristo
  conflict   runs/my-run2/dataset
  conflict   runs/my-run2/pretraining

2 conflict, 4 declared, 5 new
BLOCKED -- 2 to resolve


In [24]:
declaration.write()
print(declaration.check())

ValueError: refusing to declare over an inconsistent tree:
conflict   runs/my-run2/dataset
conflict   runs/my-run2/pretraining

Declared, not yet produced -- `declared` rows have a manifest but no files
yet. To actually run the jobs, see `demo.ipynb`'s job-list and `Job.run()`
cells, or drive `job_list(resolve(pretraining))` by hand.